In [6]:
import json, random, torch
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
actor_pipe = pipeline("text-generation", model=model_name, tokenizer=model_name, torch_dtype=torch.bfloat16, device_map="auto")
evaluator_pipe = pipeline("text-generation", model=model_name, tokenizer=model_name, torch_dtype=torch.bfloat16, device_map="auto")
reflect_pipe = pipeline("text-generation", model=model_name, tokenizer=model_name, torch_dtype=torch.bfloat16, device_map="auto")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

Device set to use cpu


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cpu


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cpu


In [8]:
def reflection_prompt(reason, previous_response):
    return f"""
మీరు అందించిన మునుపటి ప్రతిస్పందనను పరిశీలించండి:

మునుపటి ప్రతిస్పందన: {previous_response}

ఈ ప్రతిస్పందనను "{reason}" కారణంగా అంచనా వేయడంలో విఫలమైంది. దయచేసి ప్రతిస్పందనను మెరుగుపరచడానికి ప్రయత్నించండి.

మెరుగుపరచబడిన ప్రతిస్పందన:
"""

In [9]:

def evaluator_prompt(seeker, response, empathy_type):
    return f"""
కింద ఉన్న సంభాషణను పరిశీలించండి:

రోగి: {seeker}

థెరపిస్టు: {response}

మీ పని: {empathy_type} సానుభూతి ఉన్నదా లేదా అంచనా వేయాలి.

కేవలం ఇలా సమాధానం ఇవ్వండి:
PASS - ఎందుకు సరైంది
FAIL - ఎందుకు తప్పు
"""


In [10]:
def generate_conversation(topic):
    """
    Reflexion-based conversation generator.
    Actor --> Evaluator --> Reflection --> Actor
    """

    # --- 1. Generate seeker text ---
    seeker_prompt = f"విషయం: {topic}\nరోగి (సమస్య చెబుతున్నాడు):"
    seeker_text = actor_pipe(seeker_prompt, max_new_tokens=60, temperature=0.8)[0]["generated_text"]
    seeker_post = seeker_text.replace(seeker_prompt, "").strip()

    # --- 2. Actor generates therapist response ---
    therapist_prompt = f"రోగి: {seeker_post}\nసానుభూతి గల థెరపిస్టు సమాధానం:"
    response = actor_pipe(therapist_prompt, max_new_tokens=80, temperature=0.8)[0]["generated_text"]
    response = response.replace(therapist_prompt, "").strip()

    # Choose empathy dimension
    empathy_type = random.choice([
        "Emotional Reaction", "Interpretation", "Exploration"
    ])

    # ---- Reflexion Loop ----
    for attempt in range(3):
        eval_inp = evaluator_prompt(seeker_post, response, empathy_type)
        evaluation = evaluator_pipe(eval_inp, max_new_tokens=60)[0]["generated_text"]

        if "PASS" in evaluation:
            break  # Good output

        # Extract fail reason
        reason = evaluation.replace("FAIL", "").strip()

        # Reflection step
        refl_inp = reflection_prompt(reason, response)
        improved = reflect_pipe(refl_inp, max_new_tokens=80)[0]["generated_text"]

        response = improved.strip()  # Actor tries again

    # After loop → save
    return {
        "topic": topic,
        "seeker_post": seeker_post,
        "response": response,
        "empathy_type": empathy_type,
        "evaluation": evaluation,
    }


In [ ]:
from pprint import pprint

topics = [
    "ఒంటరితనం", "ఆందోళన", "దుఃఖం", "ప్రేమ విరహం",
    "పని ఒత్తిడి", "కుటుంబ సమస్యలు", "స్వీయ విశ్వాసం",
    "ఆరోగ్యం", "మానసిక శాంతి"
]

dataset = []
save_every = 10
file_index = 1

for i in range(50):
    topic = random.choice(topics)
    item = generate_conversation(topic)
    dataset.append(item)

    # 👉 Print every 5 samples (or change to whatever you like)
    if (i + 1) % 5 == 0:
        print("\n==================== SAMPLE OUTPUT ====================")
        print(f"Index: {i+1}")
        print(f"Topic: {item['topic']}")
        print(f"\n🧍‍♂️ Seeker:\n{item['seeker_post']}")
        print(f"\n👨‍⚕️ Therapist:\n{item['response']}")
        print(f"\n❤️ Empathy Type: {item['empathy_type']}")
        print(f"\n📘 Evaluation Result:\n{item['evaluation']}")
        print("=======================================================\n")

    # 👉 Save every 10 samples
    if len(dataset) % save_every == 0:
        filename = f"telugu_empathy_reflexion_{file_index}.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(dataset, f, ensure_ascii=False, indent=2)

        print(f"💾 Saved batch #{file_index} --> {filename}\n")
        dataset = []
        file_index += 1


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
